# Need Score & Candidate Location Pipeline

Computes a composite need score per census tract and identifies candidate locations
for new library branches using spatial clustering of underserved areas.

In [ ]:
import os
import kfp
from kfp import dsl, kubernetes
from kfp.dsl import Output, Dataset

## Pipeline Component

In [ ]:
RUNTIME_IMAGE = "image-registry.openshift-image-registry.svc:5000/redhat-ods-applications/runtime-datascience:datascience"


@dsl.component(
    base_image=RUNTIME_IMAGE,
    packages_to_install=["boto3", "pandas", "pyarrow", "scikit-learn", "shapely", "numpy"],
    pip_index_urls=["https://pypi.org/simple"],
)
def compute_need_scores(
    w_coverage_gap: float = 0.35,
    w_poverty: float = 0.25,
    w_no_internet: float = 0.20,
    w_pop_density: float = 0.20,
    need_threshold: float = 0.6,
    n_clusters: int = 5,
    need_scores_out: Output[Dataset] = None,
    candidates_out: Output[Dataset] = None,
):
    import io
    import os
    import logging
    import numpy as np
    import pandas as pd
    import boto3
    from shapely import wkt
    from sklearn.cluster import KMeans

    logging.basicConfig(level=logging.INFO)
    logger = logging.getLogger("need_scores")

    s3 = boto3.client(
        "s3",
        endpoint_url=os.environ["AWS_S3_ENDPOINT"],
        aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
        aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
        verify=False,
    )
    bucket = os.environ["AWS_S3_BUCKET"]

    # --- Load coverage scores ---
    obj = s3.get_object(Bucket=bucket, Key="results/coverage_scores.parquet")
    scores = pd.read_parquet(io.BytesIO(obj["Body"].read()))
    scores = scores[(scores["mode"] == "driving") & (scores["travel_time"] == 5)]
    logger.info(f"Loaded {len(scores)} coverage scores (driving/5min)")

    # --- Load census demographics ---
    obj = s3.get_object(Bucket=bucket, Key="clean/census_demos.csv")
    demos = pd.read_csv(io.BytesIO(obj["Body"].read()))
    demos = demos[["geoid", "Percent Ages 18-64: Living in Poverty", "Percent of People without Internet Access"]].copy()
    demos.columns = ["geoid", "poverty_rate", "no_internet"]
    demos["poverty_rate"] = pd.to_numeric(demos["poverty_rate"], errors="coerce").fillna(0)
    demos["no_internet"] = pd.to_numeric(demos["no_internet"], errors="coerce").fillna(0)
    logger.info(f"Loaded {len(demos)} census demo records")

    # --- Load population demographics ---
    obj = s3.get_object(Bucket=bucket, Key="clean/population_demographics.csv")
    pop = pd.read_csv(io.BytesIO(obj["Body"].read()))
    pop = pop[["geoid", "total population", "population density (per sq. mile)", "geometry"]].copy()
    pop.columns = ["geoid", "total_population", "pop_density", "geometry"]
    pop["total_population"] = pd.to_numeric(pop["total_population"], errors="coerce").fillna(0)
    pop["pop_density"] = pd.to_numeric(pop["pop_density"], errors="coerce").fillna(0)
    logger.info(f"Loaded {len(pop)} population records")

    # --- Merge all on geoid ---
    merged = pop.merge(demos, on="geoid", how="left")
    merged = merged.merge(scores[["geoid", "percent_covered"]], on="geoid", how="left")
    merged["coverage_gap"] = 1 - merged["percent_covered"].fillna(0)
    merged["poverty_rate"] = merged["poverty_rate"].fillna(0)
    merged["no_internet"] = merged["no_internet"].fillna(0)
    logger.info(f"Merged dataset: {len(merged)} tracts")

    # --- Min-max normalize ---
    def min_max_norm(series):
        mn, mx = series.min(), series.max()
        if mx - mn == 0:
            return pd.Series(0.0, index=series.index)
        return (series - mn) / (mx - mn)

    merged["norm_coverage_gap"] = min_max_norm(merged["coverage_gap"])
    merged["norm_poverty"] = min_max_norm(merged["poverty_rate"])
    merged["norm_no_internet"] = min_max_norm(merged["no_internet"])
    merged["norm_pop_density"] = min_max_norm(merged["pop_density"])

    # --- Normalize weights ---
    total_w = w_coverage_gap + w_poverty + w_no_internet + w_pop_density
    wg = w_coverage_gap / total_w
    wp = w_poverty / total_w
    wi = w_no_internet / total_w
    wd = w_pop_density / total_w

    # --- Composite need score ---
    merged["need_score"] = (
        wg * merged["norm_coverage_gap"] +
        wp * merged["norm_poverty"] +
        wi * merged["norm_no_internet"] +
        wd * merged["norm_pop_density"]
    )
    logger.info(f"Need score stats: mean={merged['need_score'].mean():.3f}, max={merged['need_score'].max():.3f}")

    # --- Write need scores to S3 ---
    score_cols = ["geoid", "need_score", "coverage_gap", "poverty_rate", "no_internet", "pop_density",
                  "norm_coverage_gap", "norm_poverty", "norm_no_internet", "norm_pop_density", "total_population"]
    need_df = merged[score_cols].copy()

    buf = io.BytesIO()
    need_df.to_parquet(buf, index=False)
    buf.seek(0)
    s3.put_object(Bucket=bucket, Key="results/need_scores.parquet", Body=buf.getvalue())
    need_df.to_parquet(need_scores_out.path, index=False)
    logger.info(f"Wrote need scores for {len(need_df)} tracts to S3")

    # --- Filter high-need tracts ---
    high_need = merged[merged["need_score"] >= need_threshold].copy()
    logger.info(f"{len(high_need)} tracts above need threshold {need_threshold}")

    if len(high_need) == 0:
        logger.warning("No tracts above threshold — writing empty candidates")
        empty = pd.DataFrame(columns=["rank", "cluster_id", "latitude", "longitude", "total_need", "num_tracts", "avg_need_score"])
        empty.to_parquet(candidates_out.path, index=False)
        buf = io.BytesIO()
        empty.to_parquet(buf, index=False)
        buf.seek(0)
        s3.put_object(Bucket=bucket, Key="results/candidate_locations.parquet", Body=buf.getvalue())
        s3.put_object(Bucket=bucket, Key="results/candidate_locations.csv", Body=empty.to_csv(index=False))
        return

    # --- Compute centroids ---
    high_need["centroid"] = high_need["geometry"].apply(lambda g: wkt.loads(g).centroid)
    high_need["lat"] = high_need["centroid"].apply(lambda c: c.y)
    high_need["lon"] = high_need["centroid"].apply(lambda c: c.x)

    # --- KMeans clustering ---
    k = min(n_clusters, len(high_need))
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    high_need["cluster"] = km.fit_predict(high_need[["lat", "lon"]].values)

    # --- Compute candidate locations ---
    candidates = []
    for cid in range(k):
        cluster = high_need[high_need["cluster"] == cid]
        weight = cluster["need_score"] * cluster["total_population"]
        total_weight = weight.sum()
        if total_weight == 0:
            clat = cluster["lat"].mean()
            clon = cluster["lon"].mean()
        else:
            clat = (cluster["lat"] * weight).sum() / total_weight
            clon = (cluster["lon"] * weight).sum() / total_weight
        candidates.append({
            "cluster_id": cid,
            "latitude": clat,
            "longitude": clon,
            "total_need": total_weight,
            "num_tracts": len(cluster),
            "avg_need_score": cluster["need_score"].mean(),
        })

    cand_df = pd.DataFrame(candidates)
    cand_df = cand_df.sort_values("total_need", ascending=False).reset_index(drop=True)
    cand_df["rank"] = cand_df.index + 1
    cand_df = cand_df[["rank", "cluster_id", "latitude", "longitude", "total_need", "num_tracts", "avg_need_score"]]

    # --- Write candidates to S3 ---
    buf = io.BytesIO()
    cand_df.to_parquet(buf, index=False)
    buf.seek(0)
    s3.put_object(Bucket=bucket, Key="results/candidate_locations.parquet", Body=buf.getvalue())
    s3.put_object(Bucket=bucket, Key="results/candidate_locations.csv", Body=cand_df.to_csv(index=False))
    cand_df.to_parquet(candidates_out.path, index=False)

    logger.info(f"Computed need scores for {len(need_df)} tracts, identified {len(cand_df)} candidate locations")

## Pipeline Definition

In [ ]:
@dsl.pipeline(
    name="cpl-need-score-candidates",
    description="Compute need scores per tract and identify candidate library locations",
)
def need_score_pipeline(
    w_coverage_gap: float = 0.35,
    w_poverty: float = 0.25,
    w_no_internet: float = 0.20,
    w_pop_density: float = 0.20,
    need_threshold: float = 0.6,
    n_clusters: int = 5,
):
    task = compute_need_scores(
        w_coverage_gap=w_coverage_gap,
        w_poverty=w_poverty,
        w_no_internet=w_no_internet,
        w_pop_density=w_pop_density,
        need_threshold=need_threshold,
        n_clusters=n_clusters,
    )
    kubernetes.use_secret_as_env(
        task,
        secret_name="cpl-s3",
        secret_key_to_env={
            "AWS_ACCESS_KEY_ID": "AWS_ACCESS_KEY_ID",
            "AWS_SECRET_ACCESS_KEY": "AWS_SECRET_ACCESS_KEY",
            "AWS_S3_ENDPOINT": "AWS_S3_ENDPOINT",
            "AWS_S3_BUCKET": "AWS_S3_BUCKET",
        },
    )

## Submit

In [ ]:
import subprocess
from kfp.compiler import Compiler
from kfp_server_api.exceptions import ApiException

token = subprocess.check_output("oc whoami -t", shell=True, text=True).strip()
pipeline_url = os.environ.get("PIPELINES_URL")

client = kfp.Client(
    host=pipeline_url,
    existing_token=token,
    verify_ssl=False,
)

Compiler().compile(need_score_pipeline, "need_score_pipeline.yaml")

try:
    client.upload_pipeline("need_score_pipeline.yaml", pipeline_name="cpl-need-score-candidates")
    print("Pipeline definition registered.")
except ApiException as e:
    if e.status == 409:
        print("Pipeline definition already exists.")
    else:
        raise

run = client.create_run_from_pipeline_func(
    need_score_pipeline,
    experiment_name="cpl-coverage",
    run_name="need-score-candidates-run",
    enable_caching=False,
)
print(f"Pipeline run submitted: {run.run_id}")